# Step 1e — Preprocessing: BIO-variables from ISIMIP3b MRI-ESM2-0 SSP585 (2061-2080) at 1deg

Future-climate counterpart to [1.1.a)_preprocessing_BIOvariables_ERA5_MIRCA2000_threshold5pct.ipynb](../1.1\)_preprocessing_biovariables_ERA5/1.1.a\)_preprocessing_BIOvariables_ERA5_MIRCA2000_threshold5pct.ipynb),
same BIO-variable definitions/ETo formula, same 1deg Europe grid (from `0.1)`), same MIRCA2000
crop calendar (held fixed -- only the climate input changes) -- built to test the SVM
(trained on ERA5 1981-2015) against GAEZ v5's own future suitability product
(`RES05-SXX`, continuous index, FP6180=2061-2080, MRI-ESM2-0, SSP585) in
`5.g)_SVM_vs_GAEZ_future_ISIMIP_threshold5pct.ipynb` / `5.g.bis)_SVM_vs_GAEZ_future_ISIMIP_no_threshold.ipynb`.

**Data**: ISIMIP3b bias-adjusted daily climate (`0_Input_data/ISIMIP_MRI-ESM2-0/ssp585/`),
native 0.5deg, variables tasmax/tasmin (K), pr (kg m-2 s-1), rsds (W m-2). Regridded to our
1deg integer-centre grid by simple 2x2 block-mean (0.5deg cells at X+-0.25 average exactly
to integer-degree centre X).

**Harvest years**: `range(2062, 2081)` -- the 2061-2080 analogue of `1.1.a)`'s
`range(1982, 2016)` for 1981-2015 (same sowing-year-before/harvest-year-of logic).


In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np
import xarray as xr
import pandas as pd
import datetime
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE       = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb
ERA5_1DEG  = BASE / "1_SVM" / "0_preprocessing_masks" / "outputs_input_data_1deg"   # grid + MIRCA2000 1deg crop masks (from 0.1)
ISIMIP_DIR = cu.data_dir(CFG, "isimip_mri_ssp585")  # data path: see 00_config_and_inventory.ipynb
OUT_ISIMIP = BASE / "1_SVM" / "1_preprocessing_BIOvariables" / "1.2)_preprocessing_biovariables_ISIMIP" / "outputs_ISIMIP_MRI-ESM2-0_ssp585_2061_2080"
OUT_ISIMIP.mkdir(parents=True, exist_ok=True)

ISIMIP_VARS = {'tasmax': 'tasmax', 'tasmin': 'tasmin', 'pr': 'pr', 'ssrd': 'rsds'}
# folder name (matches ERA5 BIO naming) -> variable name inside the ISIMIP netCDF files
YEARS_CHUNKS = ['2061_2070', '2071_2080']

print(f'ISIMIP dir : {ISIMIP_DIR}  (exists: {ISIMIP_DIR.exists()})')
print(f'ERA5 1deg dir (grid+masks): {ERA5_1DEG}  (exists: {ERA5_1DEG.exists()})')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PARAMÈTRES — modifier ici, puis relancer les cellules marquées [RUN]  ║
# ╚══════════════════════════════════════════════════════════════════════════╗
CROP_NAME  = 'Maize'
# 'Wheat-spring' | 'Wheat-winter' | 'Maize' | 'Potato'
IRRIG_TYPE = 'rf'

season_label = f'{CROP_NAME}_{IRRIG_TYPE}'
print(f"Crop label: {season_label}")

## 1. Target grid (from 0.1's ERA5 1deg output, same grid as the ERA5 BIO variables)

In [ ]:
sample_file = ERA5_1DEG / "ERA5_1deg_tasmax_1981_2015_daily.nc"
if not sample_file.exists():
    raise FileNotFoundError(f"{sample_file} missing -- run 0.1)_preprocessing_input_data.ipynb first.")
ds_grid = xr.open_dataset(sample_file)
da_ref  = ds_grid['tasmax']
target_lats_asc = da_ref.lat.values
target_lons     = da_ref.lon.values
land_mask = np.isfinite(da_ref.isel(time=0).values)
ds_grid.close()

nlat, nlon = len(target_lats_asc), len(target_lons)
print(f"Target grid: lat {target_lats_asc[0]:.1f}->{target_lats_asc[-1]:.1f} (n={nlat}), "
      f"lon {target_lons[0]:.1f}->{target_lons[-1]:.1f} (n={nlon})")
print(f"Land pixels: {land_mask.sum()} / {land_mask.size}")

## 2. Load ISIMIP 0.5deg daily (2061-2080), subset to Europe, regrid to 1deg

In [ ]:
LAT_HI, LAT_LO = target_lats_asc[-1] + 0.25, target_lats_asc[0] - 0.25   # 71.25, 34.75
LON_LO, LON_HI = target_lons[0] - 0.25, target_lons[-1] + 0.25             # -25.25, 44.25


def load_isimip_1deg(folder, varname):
    """ISIMIP3b 0.5deg daily -> our 1deg integer-centre grid, Europe window only.
    2x2 block-mean: 0.5deg cells sit at integer+-0.25, so each pair averages exactly to
    the integer centre -- no interpolation, a true windowed area-mean like elsewhere in
    this pipeline."""
    das = []
    for chunk in YEARS_CHUNKS:
        f = ISIMIP_DIR / folder / f"mri-esm2-0_r1i1p1f1_w5e5_ssp585_{varname}_global_daily_{chunk}.nc"
        ds = xr.open_dataset(f)
        da = ds[varname].sel(lat=slice(LAT_HI, LAT_LO), lon=slice(LON_LO, LON_HI))
        das.append(da.load())
        ds.close()
        print(f"    loaded {chunk} ({folder})")
    da_full = xr.concat(das, dim='time')
    assert da_full.sizes['lat'] == 2 * nlat and da_full.sizes['lon'] == 2 * nlon, \
        f"unexpected subset size: {da_full.sizes}"
    da_coarse = da_full.coarsen(lat=2, lon=2, boundary='exact').mean()
    da_coarse = da_coarse.sortby('lat').assign_coords(lat=target_lats_asc, lon=target_lons)
    return da_coarse


print("Loading + regridding ISIMIP MRI-ESM2-0 SSP585 2061-2080 (tasmax)...")
tmax_K = load_isimip_1deg('tasmax', 'tasmax')
print("Loading + regridding (tasmin)...")
tmin_K = load_isimip_1deg('tasmin', 'tasmin')
print("Loading + regridding (pr)...")
pr_kgm2s = load_isimip_1deg('pr', 'pr')
print("Loading + regridding (rsds)...")
sw_Wm2 = load_isimip_1deg('ssrd', 'rsds')

# Unit conversions -- K -> degC, kg m-2 s-1 -> mm/day, W/m2 -> MJ/m2/day (same 0.0864 factor as ERA5)
tmax_C = tmax_K - 273.15
tmin_C = tmin_K - 273.15
pr_mm  = pr_kgm2s * 86400.0
sw_MJ  = sw_Wm2 * 0.0864

times = tmax_C.time.values
print(f"\nGrid: {nlat} lat x {nlon} lon, {len(times)} days")
print(f"Period: {str(times[0])[:10]} -> {str(times[-1])[:10]}")

s = dict(time='2070-07-15', method='nearest')
print(f"\nSanity check (Jul 2070, lat~40N, lon~15E):")
print(f"  Tmax : {float(tmax_C.sel(**s).sel(lat=40.0, lon=15.0, method='nearest')):.1f} degC")
print(f"  Pr   : {float(pr_mm.sel(**s).sel(lat=40.0, lon=15.0, method='nearest')):.2f} mm/day")
print(f"  Solar: {float(sw_MJ.sel(**s).sel(lat=40.0, lon=15.0, method='nearest')):.1f} MJ/m2/day")

## 3. Crop calendar (MIRCA2000, held fixed -- same as the ERA5 notebook)

In [ ]:
def fill_calendar_nn(cal_arr, land_mask, k=5):
    from scipy.spatial import cKDTree
    LA, LO = np.meshgrid(target_lats_asc, target_lons, indexing='ij')
    X = np.column_stack([LA.ravel(), (LO * np.cos(np.radians(55.0))).ravel()])
    known   = (~np.isnan(cal_arr) & land_mask).ravel()
    to_fill = (np.isnan(cal_arr) & land_mask).ravel()
    if not known.any() or not to_fill.any(): return cal_arr
    kk = min(k, int(known.sum()))
    d, ix = cKDTree(X[known]).query(X[to_fill], k=kk)
    d = np.maximum(d.reshape(-1, kk), 0.5); ix = ix.reshape(-1, kk)
    w = 1.0 / d**2
    vals = cal_arr.ravel()[known][ix]
    out = cal_arr.copy().ravel()
    out[to_fill] = (w * vals).sum(1) / w.sum(1)
    return out.reshape(cal_arr.shape)


crop_mask_nc = ERA5_1DEG / f"mirca2000_1deg_{season_label}.nc"
print(f"  {'OK' if crop_mask_nc.exists() else 'MISSING'} : {crop_mask_nc.name}")
cm = xr.open_dataset(crop_mask_nc)
pres_1deg = cm['presence'].values.astype(float)
sowing_doy = cm['sowing_doy'].values.astype(float)
maturity_doy = cm['maturity_doy'].values.astype(float)
cm.close()

crop_present_mask = np.where(land_mask, pres_1deg, np.nan)
print(f"  Presence cells in window: {int(np.nansum(crop_present_mask == 1))}")
anthesis_doy = maturity_doy - 60

sowing_doy_full   = fill_calendar_nn(sowing_doy,   land_mask)
maturity_doy_full = fill_calendar_nn(maturity_doy, land_mask)
anthesis_doy_full = maturity_doy_full - 60
print(f"  Land pixels after NN fill: {int(np.sum(~np.isnan(sowing_doy_full) & land_mask))}")

## 4. ETo — Hargreaves method (identical to `1.1.a)`, including the Ra/lambda fix)

In [ ]:
def compute_Ra(lat_rad_2d, doy):
    Gsc   = 0.0820
    dr    = 1 + 0.033 * np.cos(2 * np.pi / 365 * doy)
    delta = 0.409 * np.sin(2 * np.pi / 365 * doy - 1.39)
    ws    = np.arccos(np.clip(-np.tan(lat_rad_2d) * np.tan(delta), -1, 1))
    return (24*60/np.pi)*Gsc*dr*(ws*np.sin(lat_rad_2d)*np.sin(delta)
                                  + np.cos(lat_rad_2d)*np.cos(delta)*np.sin(ws))


def compute_ETo_day(tmax_day, tmin_day, lat_rad_2d, doy):
    Tmean = (tmax_day + tmin_day) / 2.0
    TD    = np.maximum(tmax_day - tmin_day, 0.0)
    lam   = 2.501 - 0.002361 * Tmean
    Ra_mm = compute_Ra(lat_rad_2d, doy) / lam
    return 0.0023 * Ra_mm * (Tmean + 17.8) * np.sqrt(TD)


print("Loading arrays into memory...")
TMAX = tmax_C.values; TMIN = tmin_C.values; PR = pr_mm.values; SW = sw_MJ.values
dates = pd.DatetimeIndex(tmax_C.time.values)
doys_all = dates.dayofyear.values
lat_rad_2d = np.radians(target_lats_asc[:, None]) * np.ones((1, nlon))

print("Precomputing ETo for all days...")
ETO = np.full_like(TMAX, np.nan)
for t in range(len(dates)):
    ETO[t] = compute_ETo_day(TMAX[t], TMIN[t], lat_rad_2d, doys_all[t])
    if t % 2000 == 0:
        print(f"  {t}/{len(dates)} days")
print("ETo done.")

## 5. Compute BIO variables (2061-2080 climatology, harvest years 2062-2080)

In [ ]:
def compute_bio_variables(TMAX, TMIN, PR, SW, ETO, dates,
                          sowing_doy, anthesis_doy, maturity_doy, harvest_years):
    """Same logic as 1.1.a)'s compute_bio_variables -- harvest_years is now a parameter
    (1.1.a) hardcodes range(1982,2016) for ERA5 1981-2015; here range(2062,2081))."""
    nlat, nlon = sowing_doy.shape
    yr_all  = dates.year.values
    doy_all = dates.dayofyear.values
    mon_all = dates.month.values
    bio_sum = {i: np.zeros((nlat, nlon)) for i in range(1, 13)}
    bio_cnt = {i: np.zeros((nlat, nlon)) for i in range(1, 13)}
    valid   = ~np.isnan(sowing_doy) & ~np.isnan(maturity_doy)
    wrap    = valid & (maturity_doy > 365)
    yrs_present = set(yr_all)

    def ylen(y): return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365

    for hy in harvest_years:
        print(f'  {hy}...', end=' ')
        if not (yr_all == hy).any(): print('skip'); continue
        idx = np.where((yr_all == hy) | (yr_all == hy - 1))[0]
        yr3  = yr_all[idx][:, None, None]
        doy3 = doy_all[idx][:, None, None]
        mon3 = mon_all[idx][:, None, None]
        t3   = doy3 + np.where(yr3 == hy, ylen(hy - 1), 0)
        shift = np.where(wrap, 0, ylen(hy - 1))[None]
        sow3 = np.nan_to_num(sowing_doy)[None] + shift
        ant3 = np.nan_to_num(anthesis_doy)[None] + shift
        mat3 = np.nan_to_num(maturity_doy)[None] + shift
        vp3  = valid[None] & (((hy - 1) in yrs_present) | ~wrap[None])

        p1 = (t3 >= sow3) & (t3 < ant3)
        p2 = (t3 >= ant3) & (t3 <= mat3)
        gs = (p1 | p2) & vp3; p1 = p1 & vp3; p2 = p2 & vp3

        tm = TMAX[idx]; tn = TMIN[idx]; pr = PR[idx]; sw = SW[idx]; et = ETO[idx]
        def pm(a, m): return np.nanmean(np.where(m, a, np.nan), axis=0)
        def acc(k, v):
            f = np.isfinite(v); bio_sum[k] += np.where(f, v, 0); bio_cnt[k] += f

        acc(1, pm(tm, p1)); acc(2, pm(tn, p1)); acc(3, pm(sw, p1))
        acc(4, pm(et, p1)); acc(5, pm(pr, p1))
        acc(6, pm(tm, p2)); acc(7, pm(tn, p2)); acc(8, pm(sw, p2))
        acc(9, pm(et, p2)); acc(10, pm(pr, p2))

        tmx = np.full((nlat, nlon), np.nan); tmn = np.full((nlat, nlon), np.nan)
        for m in range(1, 13):
            mm = gs & (mon3 == m)
            if not np.any(mm): continue
            tmx = np.fmax(tmx, np.nanmean(np.where(mm, tm, np.nan), axis=0))
            tmn = np.fmin(tmn, np.nanmean(np.where(mm, tn, np.nan), axis=0))
        acc(11, tmx); acc(12, tmn)
        print('ok')

    bio = {}
    for i in range(1, 13):
        with np.errstate(invalid='ignore', divide='ignore'):
            bio[i] = np.where(bio_cnt[i] > 0, bio_sum[i] / bio_cnt[i], np.nan)
        bio[i][~valid] = np.nan
    return bio


print(f"Computing BIO variables -- {season_label} (ISIMIP MRI-ESM2-0 SSP585 2061-2080)...")
bio = compute_bio_variables(
    TMAX, TMIN, PR, SW, ETO, dates,
    sowing_doy_full, anthesis_doy_full, maturity_doy_full,
    harvest_years=range(2062, 2081)
)
print("\nDone:")
for i in range(1, 13):
    n_valid = int(np.sum(~np.isnan(bio[i])))
    print(f"  BIO{i:02d}: {np.nanmin(bio[i]):.2f} -- {np.nanmax(bio[i]):.2f}  ({n_valid} valid pixels)")

## 6. Save BIO variables

In [ ]:
bio_names = {
    1:'Avg Tmax sowing-anthesis [degC]',      2:'Avg Tmin sowing-anthesis [degC]',
    3:'Avg solar rad sowing-anthesis [MJ/m2/d]', 4:'Avg ETo sowing-anthesis [mm/d]',
    5:'Avg precip sowing-anthesis [mm/d]',    6:'Avg Tmax anthesis-maturity [degC]',
    7:'Avg Tmin anthesis-maturity [degC]',    8:'Avg solar rad anthesis-maturity [MJ/m2/d]',
    9:'Avg ETo anthesis-maturity [mm/d]',     10:'Avg precip anthesis-maturity [mm/d]',
    11:'Tmax warmest month sowing-maturity [degC]',
    12:'Tmin coldest month sowing-maturity [degC]',
}
data_vars = {}
for i in range(1, 13):
    arr = bio[i].copy(); arr[~land_mask] = np.nan
    data_vars[f'BIO{i:02d}'] = xr.DataArray(
        arr, dims=['lat', 'lon'],
        coords={'lat': target_lats_asc, 'lon': target_lons},
        attrs={'long_name': bio_names[i]}
    )
ds_bio = xr.Dataset(data_vars)
ds_bio.attrs['description'] = (
    f'BIO variables, ISIMIP3b MRI-ESM2-0 SSP585 2061-2080, 1deg grid (integer centres), '
    f'Europe land, crop={season_label}'
)
out_nc = OUT_ISIMIP / f'BIOvariables_ISIMIP_MRI-ESM2-0_ssp585_2061_2080_1deg_{season_label}.nc'
ds_bio.to_netcdf(out_nc)
print(f"Saved: {out_nc}")